# EDA silver layer

In [0]:

df = spark.sql("SELECT * FROM marathos.bronze.raw_marathos")

display(df)

In [0]:
df_country_code = spark.sql("FROM marathos.bronze.raw_country_code")
df_country_code.display()

In [0]:
from Marathos.utils.utils import rename_columns_to_snake_case
df = rename_columns_to_snake_case(df)
df.display()

In [0]:
from Marathos.utils.utils import rename_columns_to_snake_case
df = rename_columns_to_snake_case(df)
df.display()

In [0]:
from pyspark.sql.functions import regexp_extract, split, when, col, lit, get, size, expr

df_valid = (
    df
    .withColumn(
        "event_unit",
        regexp_extract(col("event_distance/length"), r"(km|mi|h)", 1)
    )
    .withColumn(
        "performance_clean",
        regexp_extract(col("athlete_performance"), r"^([\d:\.]+)", 1)
    )
    .withColumn(
        "performance_hours",
        when(
            col("event_unit").isin("km", "mi"),
            when(
                size(split(col("performance_clean"), ":")) == 3,
                expr("try_cast(split(performance_clean, ':')[0] AS FLOAT)") +
                expr("try_cast(split(performance_clean, ':')[1] AS FLOAT)") / 60 +
                expr("try_cast(split(performance_clean, ':')[2] AS FLOAT)") / 3600
            ).when(
                size(split(col("performance_clean"), ":")) == 2,
                expr("try_cast(split(performance_clean, ':')[0] AS FLOAT)") +
                expr("try_cast(split(performance_clean, ':')[1] AS FLOAT)") / 60
            ).otherwise(lit(None))
        ).otherwise(lit(None))
    )

    .withColumn(
        "performance_km",
        when(
            col("event_unit") == "h",
            expr("try_cast(performance_clean AS FLOAT)")
        ).otherwise(lit(None))
    )
    .drop("athlete_performance")
)

df_valid.select(
    "event_distance/length", "event_unit", "performance_hours", "performance_km"
).display()

In [0]:
from pyspark.sql.functions import regexp_extract, split, when, col, lit, size, expr, regexp_replace

df_valid = (
    df
    # Extrahera enhet (km, mi, h)
    .withColumn(
        "event_unit",
        regexp_extract(col("event_distance/length"), r"(km|mi|h)", 1)
    )

    # Klassificera eventtyp
    .withColumn(
        "event_type",
        when(col("event_distance/length").rlike(r"Etappen"), "multi_stage")
        .when(col("event_distance/length").rlike(r"^\d+d$"), "time_based")
        .when(col("event_unit") == "h", "time_based")
        .when(col("event_unit").isin("km", "mi"), "distance_based")
        .otherwise("unknown")
    )

    # Extrahera numeriskt avstånd/tid
    .withColumn(
        "event_distance_numeric",
        regexp_extract(col("event_distance/length"), r"^([\d.]+)", 1).cast("float")
    )

    # Rensa prestation
    .withColumn(
        "performance_clean",
        regexp_extract(col("athlete_performance"), r"^([\d:\.]+)", 1)
    )

    # Omvandla tid till timmar (distance_based)
    .withColumn(
        "performance_hours",
        when(
            col("event_unit").isin("km", "mi"),
            when(
                size(split(col("performance_clean"), ":")) == 3,
                expr("try_cast(split(performance_clean, ':')[0] AS FLOAT)") +
                expr("try_cast(split(performance_clean, ':')[1] AS FLOAT)") / 60 +
                expr("try_cast(split(performance_clean, ':')[2] AS FLOAT)") / 3600
            ).when(
                size(split(col("performance_clean"), ":")) == 2,
                expr("try_cast(split(performance_clean, ':')[0] AS FLOAT)") +
                expr("try_cast(split(performance_clean, ':')[1] AS FLOAT)") / 60
            ).otherwise(lit(None))
        ).otherwise(lit(None))
    )

    # Extrahera distans i km (time_based)
    .withColumn(
        "performance_km",
        when(
            col("event_unit") == "h",
            expr("try_cast(performance_clean AS FLOAT)")
        ).otherwise(lit(None))
    )

    .drop("athlete_performance")
)

df_valid.select(
    "event_distance/length",
    "event_unit",
    "event_type",
    "event_distance_numeric",
    "performance_hours",
    "performance_km"
).display()

In [0]:
df_valid.select("athlete_country").display()

In [0]:
df_valid = (
    df_valid
    .join(
        df_country_code.select("country_code", "country_name"),
        df_valid["athlete_country"] == df_country_code["country_code"],
        how="left"
    )
    .drop("country_code")
    .withColumnRenamed("country_name", "athlete_country_name")
)


split race name from country code

In [0]:
from pyspark.sql.functions import regexp_extract, regexp_replace, col

df_valid = (
    df_valid

    .withColumn("event_country", regexp_extract(col("event_name"), r"\(([^)]+)\)$", 1))

    # Ta bort landet + parentes från event-namnet
    .withColumn("event_name_clean", regexp_replace(col("event_name"), r"\s*\([^)]+\)$", ""))
    .drop("event_name")
)

df_valid.display()


In [0]:
df_valid.select("athlete_average_speed").show(20, False)

In [0]:
df_valid.printSchema()

In [0]:
from pyspark.sql.functions import col

df_valid = df_valid.withColumn(
    "athlete_avg_speed_kmh",
    expr("try_cast(athlete_average_speed as double)")
)

df_valid.select("athlete_avg_speed_kmh").show(20, False)

In [0]:

df_filtered = df_valid.filter(~col("event_distance/length").rlike(r"^\d+d$"))

print("Innan:", df_valid.count())
print("Efter:", df_filtered.count())
print("Borttagna d-rader:", df_valid.count() - df_filtered.count())

In [0]:
df_invalid_performance = df_valid.filter(
    ~(col("event_unit").isin("km", "mi") & col("performance_hours").isNull()) &
    ~((col("event_unit") == "h") & col("performance_km").isNull())  # ← parentes runt ==
)


print("Ogiltiga performance-rader:", df_invalid_performance.count())
df_invalid_performance.select("event_distance/length", "event_unit", "performance_hours", "performance_km").display()

In [0]:
df_valid.select("event_dates").distinct().display()

In [0]:
from pyspark.sql.functions import when, col

df_valid.withColumn(
    "date_format_type",
    when(col("event_dates").rlike(r"^\d{2}\.\d{2}\.\d{4}$"), "enkelt_datum")
    .when(col("event_dates").rlike(r"\d{2}\.-\d{2}\.\d{2}\.\d{4}"), "intervall")
    .otherwise("okänt_format")
).groupBy("date_format_type").count().display()

In [0]:
from pyspark.sql.functions import (
    regexp_extract, when, col, lit, concat, expr
)

df_valid = df_valid.withColumn(
    "event_start_date",
    when(
        # Enkelt datum: "07.12.1991"
        col("event_dates").rlike(r"^\d{2}\.\d{2}\.\d{4}$"),
        expr("try_to_date(event_dates, 'dd.MM.yyyy')")
    ).when(
        # Intervall: "23.-24.11.1991"
        col("event_dates").rlike(r"^\d{2}\.-\d{2}\.\d{2}\.\d{4}$"),
        expr("""
            try_to_date(
                concat(
                    regexp_extract(event_dates, '^(\\d{2})\\.', 1),
                    '.',
                    regexp_extract(event_dates, '^\\d{2}\\.-\\d{2}\\.(\\d{2}\\.\\d{4})$', 1)
                ),
                'dd.MM.yyyy'
            )
        """)
    ).otherwise(lit(None))
).show()

felsöka mina datum som blir fel

In [0]:
#funkade inte att använda gamla df. Något har trasslat till det

df = rename_columns_to_snake_case(
    spark.sql("SELECT * FROM marathos.bronze.raw_marathos")
)
df.select("event_dates").distinct().show(50, False)

In [0]:
df = df.withColumn(
    "event_start_date",
    # Enkelt datum: 01.05.1991
    when(
        col("event_dates").rlike(r"^\d{2}\.\d{2}\.\d{4}$"),
        expr("try_to_date(event_dates, 'dd.MM.yyyy')")
    )
    # Spann variant 1: 01.-02.11.1991 (ta dag + MM.yyyy från slutet)
    .when(
        col("event_dates").rlike(r"^\d{2}\.-\d{2}\.\d{2}\.\d{4}$"),
        expr("""
            try_to_date(
                concat(
                    regexp_extract(event_dates, '^(\\d{2})\\.-', 1),
                    '.',
                    regexp_extract(event_dates, '\\d{2}\\.(\\d{2}\\.\\d{4})$', 1)
                ),
                'dd.MM.yyyy'
            )
        """)
    )
    # Spann variant 2: 30.04.-01.05.2016 (ta första datumet + år från slutet)
    .when(
        col("event_dates").rlike(r"^\d{2}\.\d{2}\.-\d{2}\.\d{2}\.\d{4}$"),
        expr("""
            try_to_date(
                concat(
                    regexp_extract(event_dates, '^(\\d{2}\\.\\d{2}\\.)', 1),
                    regexp_extract(event_dates, '(\\d{4})$', 1)
                ),
                'dd.MM.yyyy'
            )
        """)
    )
    .otherwise(lit(None))
)

df.select("event_dates", "event_start_date").display()

In [0]:
from pyspark.sql.functions import regexp_extract, col

df.select(
    "event_dates",
    regexp_extract(col("event_dates"), r"^\d{2}\.\d{2}\.\d{4}$", 0).alias("match_simple"),
    regexp_extract(col("event_dates"), r"^\d{2}\.-\d{2}\.\d{2}\.\d{4}$", 0).alias("match_variant1"),
    regexp_extract(col("event_dates"), r"^\d{2}\.\d{2}\.-\d{2}\.\d{2}\.\d{4}$", 0).alias("match_variant2"),
).show(50, False)

In [0]:
df.select(
    col("event_dates"),
    expr("length(event_dates)").alias("len"),
    expr("hex(event_dates)").alias("hex")
).filter(col("event_dates") == "01.05.1991").show(1, False)

In [0]:
from pyspark.sql.functions import regexp_extract, col

df.filter(col("event_dates").contains("-")).select(
    "event_dates",
    regexp_extract(col("event_dates"), r"^\d{2}\.-\d{2}\.\d{2}\.\d{4}$", 0).alias("match_variant1"),
    regexp_extract(col("event_dates"), r"^\d{2}\.\d{2}\.-\d{2}\.\d{2}\.\d{4}$", 0).alias("match_variant2"),
).show(20, False)

In [0]:
df.filter(col("event_dates").contains("-")).select(
    "event_dates",
    expr("""
        to_date(
            concat(
                regexp_extract(event_dates, '^(\\d{2})\\.-', 1),
                '.',
                regexp_extract(event_dates, '\\d{2}\\.(\\d{2}\\.\\d{4})$', 1)
            ),
            'dd.MM.yyyy'
        )
    """).alias("parsed_date")
).show(20, False)

In [0]:
df.filter(col("event_dates").contains("-")).select(
    "event_dates",
    regexp_extract(col("event_dates"), r"^(\d{2})\.-", 1).alias("dag"),
    regexp_extract(col("event_dates"), r"\d{2}\.(\d{2}\.\d{4})$", 1).alias("manad_ar")
).show(20, False)

In [0]:
df.filter(col("event_dates").contains("-")).select(
    "event_dates",
    expr("""
        try_to_date(
            concat(
                regexp_extract(event_dates, '^(\\d{2})\\.-', 1),
                '.',
                regexp_extract(event_dates, '\\d{2}\\.(\\d{2}\\.\\d{4})$', 1)
            ),
            'dd.MM.yyyy'
        )
    """).alias("parsed_date")
).show(20, False)

In [0]:
df.filter(col("event_dates").contains("-")).select(
    "event_dates",
    expr("""
        concat(
            regexp_extract(event_dates, '^(\\d{2})\\.-', 1),
            '.',
            regexp_extract(event_dates, '\\d{2}\\.(\\d{2}\\.\\d{4})$', 1)
        )
    """).alias("concatenated")
).show(20, False)

In [0]:
df.filter(col("event_dates").contains("-")).select(
    "event_dates",
    expr("""
        try_to_date(
            concat(
                regexp_extract(event_dates, '^(\\d{2})\\.-', 1),
                '.',
                regexp_extract(event_dates, '\\d{2}\\.(\\d{2}\\.\\d{4})$', 1)
            ),
            'dd.MM.yyyy'
        )
    """).alias("parsed_date"),
    expr("""
        concat(
            regexp_extract(event_dates, '^(\\d{2})\\.-', 1),
            '.',
            regexp_extract(event_dates, '\\d{2}\\.(\\d{2}\\.\\d{4})$', 1)
        )
    """).alias("concatenated")
).show(20, False)

In [0]:
df.filter(col("event_dates").contains("-")).select(
    "event_dates",
    regexp_extract(col("event_dates"), r"^(\d{2})\.-", 1).alias("dag"),
    regexp_extract(col("event_dates"), r"^(\d{2})", 1).alias("dag_enkel"),
).show(5, False)

In [0]:
df.filter(col("event_dates").contains("-")).select(
    "event_dates",
    to_date(
        concat(
            regexp_extract(col("event_dates"), r"^(\d{2})", 1),
            lit("."),
            regexp_extract(col("event_dates"), r"\.(\d{2}\.\d{4})$", 1)
        ),
        "dd.MM.yyyy"
    ).alias("parsed_date")
).show(20, False)

löst problemet

In [0]:
df = df.withColumn(
    "event_start_date",
    # Enkelt datum: 01.05.1991
    when(
        col("event_dates").rlike(r"^\d{2}\.\d{2}\.\d{4}$"),
        to_date(col("event_dates"), "dd.MM.yyyy")
    )
    # Spann variant 1: 23.-24.11.1991
    .when(
        col("event_dates").rlike(r"^\d{2}\.-\d{2}\.\d{2}\.\d{4}$"),
        to_date(
            concat(
                regexp_extract(col("event_dates"), r"^(\d{2})", 1),
                lit("."),
                regexp_extract(col("event_dates"), r"\.(\d{2}\.\d{4})$", 1)
            ),
            "dd.MM.yyyy"
        )
    )
    # Spann variant 2: 30.04.-01.05.2016
    .when(
        col("event_dates").rlike(r"^\d{2}\.\d{2}\.-\d{2}\.\d{2}\.\d{4}$"),
        to_date(
            concat(
                regexp_extract(col("event_dates"), r"^(\d{2}\.\d{2}\.)", 1),
                regexp_extract(col("event_dates"), r"(\d{4})$", 1)
            ),
            "dd.MM.yyyy"
        )
    )
    .otherwise(lit(None))
)

df.select("event_dates", "event_start_date").display()
